# CI/CD for ML — Examples

This notebook demonstrates the logic behind CI checks locally: unit tests, a model quality gate, an artifact manifest, and a simple release decision. It does not run GitHub Actions or deploy a service. A workflow YAML example is included in `README.md`.

Install dependencies:

```bash
pip install numpy pandas scikit-learn pytest
```

In [ ]:
import json, hashlib, tempfile, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import joblib

ROOT = Path(tempfile.mkdtemp(prefix="ml-cicd-demo-"))
print("Temporary workspace:", ROOT)
print("scikit-learn:", sklearn.__version__)

## 1. Simulate a unit-test stage

In [ ]:
def safe_ratio(numerator, denominator):
    if denominator == 0:
        raise ValueError("denominator cannot be zero")
    return numerator / denominator

assert safe_ratio(6, 3) == 2
try:
    safe_ratio(1, 0)
except ValueError:
    pass
else:
    raise AssertionError("Expected zero denominator to fail")
print("Unit checks passed.")

## 2. Validate a synthetic dataset schema

In [ ]:
X_arr, y = make_classification(
    n_samples=1000, n_features=6, n_informative=4, n_redundant=1,
    weights=[0.68, 0.32], random_state=42
)
feature_names = [f"feature_{i}" for i in range(X_arr.shape[1])]
X = pd.DataFrame(X_arr, columns=feature_names)
required_columns = set(feature_names)
assert required_columns.issubset(X.columns)
assert X[feature_names].notna().all().all()
assert len(X) == len(y)
print("Schema checks passed:", X.shape)

## 3. Train and evaluate a candidate model
The example uses a held-out split and keeps preprocessing inside the pipeline.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
}
metrics

## 4. Apply an explicit quality gate
These thresholds are illustrative. In a real project, define them with stakeholders before inspecting the final test results.

In [ ]:
QUALITY_GATE = {"minimum_recall": 0.70, "minimum_f1": 0.65}

def quality_gate(metrics, thresholds=QUALITY_GATE):
    return (
        metrics.get("recall", -1) >= thresholds["minimum_recall"]
        and metrics.get("f1", -1) >= thresholds["minimum_f1"]
    )

gate_passed = quality_gate(metrics)
print("Metrics:", metrics)
print("Quality gate passed:", gate_passed)

## 5. Save an artifact and create traceability metadata

In [ ]:
artifact_path = ROOT / "candidate_model.joblib"
joblib.dump(model, artifact_path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "model_name": "demo-classifier",
    "model_version": "0.1.0",
    "artifact_file": artifact_path.name,
    "artifact_sha256": sha256_file(artifact_path),
    "source_commit": "replace-with-real-git-commit",
    "data_version": "synthetic-seed-42",
    "feature_schema_version": "features-v1",
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "metrics": metrics,
    "quality_gate_passed": bool(gate_passed),
    "release_status": "candidate",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
manifest_path = ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path.read_text(encoding="utf-8"))

## 6. Verify the artifact and run a smoke test

In [ ]:
assert sha256_file(artifact_path) == manifest["artifact_sha256"]
# This artifact was created by this notebook in the current temporary directory.
loaded_model = joblib.load(artifact_path)
sample_predictions = loaded_model.predict(X_test.iloc[:5])
assert len(sample_predictions) == 5
assert set(np.unique(sample_predictions)).issubset({0, 1})
print("Artifact integrity and smoke test passed.")

## 7. Simulate a release decision

In [ ]:
if gate_passed:
    manifest["release_status"] = "eligible-for-staging"
else:
    manifest["release_status"] = "rejected"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Release status:", manifest["release_status"])
print("In production, eligibility would also require tests, security scans, approvals, and deployment checks.")

## 8. Example CI workflow outline

```yaml
name: ML CI
on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
permissions:
  contents: read
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -r requirements.txt
      - run: pip install pytest ruff
      - run: ruff check .
      - run: pytest -q
```

For higher-assurance workflows, pin actions to reviewed commit SHAs. Add model/data checks, security scans, artifact publishing, and protected deployment environments as required. Do not expose production secrets to untrusted pull-request code.

## Key takeaways

- CI should fail clearly when code, data, API, or model checks fail.
- Evaluation results must be tied to a known dataset and protocol.
- A model artifact should carry source/data/schema lineage and an integrity digest.
- Passing CI is not automatic permission to deploy to production.
- Use minimal permissions, protect secrets, monitor releases, and keep a tested rollback target.